<a href="https://colab.research.google.com/github/irum-zahra-awan/geneai/blob/main/devfest_creating_marketing_assets_gemini_2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Copyright 2024 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Creating Marketing Assets using Gemini 2.0

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/irum-zahra-awan/geneai/blob/main/devfest_creating_marketing_assets_gemini_2_0.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/irum-zahra-awan/geneai/blob/main/devfest_creating_marketing_assets_gemini_2_0.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>


| Author |
| --- |
| [Irum Zahra](https://github.com/irum-zahra-awan/) |

## Overview

The new Google Gen AI SDK provides a unified interface to Gemini 2.0 through both the Gemini Developer API and the Gemini API on Vertex AI. With a few exceptions, code that runs on one platform will run on both. This means that you can prototype an application using the Developer API and then migrate the application to Vertex AI without rewriting your code.

In this tutorial, you will learn how to combine the multimodal capabilities of Gemini and Grounding with Google Search to create a marketing campaign brief and marketing assets.

You will complete the following tasks:
- Get started with the unified Google Gen AI SDK
- Create a marketing campaign brief and assets with Gemini, Grounding with Google Search and Controlled Generation

## Get started

### Install Google Gen AI SDK


In [1]:
%pip install --upgrade --quiet google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 883.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.1/719.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.9/234.9 kB 12.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.43.0, but you have google-auth 2.47.0 which is incompatible.


### Authenticate your notebook environment (Colab only)

If you're running this notebook on Google Colab, run the cell below to authenticate your environment.

In [2]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

## Using the Google Gen AI SDK

### Import the Google Gen AI SDK and other required libraries

In [3]:
import json
import os

from IPython.display import Markdown, display
from google import genai
from google.genai import types
from google.genai.types import GenerateContentConfig, GoogleSearch, Tool
from pydantic import BaseModel

### Using Gemini 2.0 Flash with Vertex AI

The new Google Gen AI SDK provides a unified interface to Gemini 2.0 Flash through both the Gemini Developer API and the Gemini API in Vertex AI. Gemini 2.0 Flash is also available through Google AI Studio and Vertex AI Studio.

- **[Gemini Developer API](https://ai.google.dev/gemini-api/docs)**: Experiment, prototype, and deploy Gen AI projects.
- **[Vertex AI](https://cloud.google.com/vertex-ai/generative-ai/docs)**: Build enterprise-ready AI projects on Google Cloud.

The Google Gen AI SDK provides a unified interface to these two API services.

#### Vertex AI

For those who are looking to build enterprise-ready AI applications in the Cloud you can use [Vertex AI](https://cloud.google.com/vertex-ai?e=48754805&hl=nl). This means that you can prototype an application using the Gemini Developer API and then migrate the application to Vertex AI without rewriting your code. In the following section, we'll take you through steps on how you can switch from Gemini Developer API to Vertex AI.

**To get started, you'll need:**
1. A Google Cloud Project
  - You can choose to [create a new Google Cloud Project](https://cloud.google.com/resource-manager/docs/creating-managing-projects#creating_a_project) OR
  - Reuse an existing project (the same one you used to generate your API key or any other existing projects)
2. To enable the [Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com)

Once your project is all set up, you're ready to switch out your code in the following section!

##### **Set Google Cloud project and location information**

**Google Cloud Projects**

In Google Cloud, projects are the fundamental building blocks for organizing, managing and securing your cloud resources. With projects, you'll be able to easily isolate resources and assign fine-grained permissions to users/service accounts at the project level. This ensures that only authorized individuals can access and manage resources, thereby enhancing security.

**Google Cloud Locations**

Location is also important as it mainly affects performance and compliance with data regulations:

- Performance: Choosing a location closer to your users reduces latency.

- Data residency: Your use case, organization or industry might constraint the location of resources.

So, selecting the right location for your Google Cloud resources is crucial for optimizing your applications and ensuring you meet your business requirements.

**In the next cell, do the following:**
- Replace ```PROJECT_ID``` with your Google Cloud project ID

In [10]:
PROJECT_ID = "devfest2025-481107"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "us-central1")

In [11]:
# Instantiate client for Vertex AI
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

## Sample Use Case - Creation and Localization of Marketing Assets

In this section of the notebook, let's take a look at how we can apply Gemini to a real-world use case.

Imagine you're part of a marketing team and you are tasked to create assets for a marketing campaign for the newest phone that your company is launching. As part of this campaign, you'll need to create the following assets:
1. Marketing Campaign Brief
2. Market Research on the industry
2. Social Media Post Ad Copy
3. Storyboard for a short-form video

Given that your company operates in multiple countries, there is also the added requirement to generate assets in multiple languages such as English, French and Japanese.

Let's see how you can use Gemini 2.0 Flash to help you accomplish these tasks!

In [12]:
MODEL_ID = "gemini-2.0-flash-001"  # @param {type: "string"}

### Creating a marketing campaign brief using a past campaign as reference

#### Let's have a look at a sample past campaign brief
Your team has done a few campaigns for phone launches in the past and documented each campaign's details in the form of a PDF document. These are stored in [Google Cloud Storage (GCS)](https://cloud.google.com/storage?e=48754805&hl=nl), a scalable, secure, and cost-effective object storage solution.

Let's have a look at one of the samples.

In [13]:
# Set the Cloud Storage path
marketing_brief_file_path = "github-repo/generative-ai/gemini2/use-cases/marketing_example/sample_marketing_campaign_brief.pdf"
marketing_brief_file_uri = f"gs://{marketing_brief_file_path}"
marketing_brief_file_url = f"https://storage.googleapis.com/{marketing_brief_file_path}"

print("Click to view the sample file:")
print(marketing_brief_file_url)

Click to view the sample file:
https://storage.googleapis.com/github-repo/generative-ai/gemini2/use-cases/marketing_example/sample_marketing_campaign_brief.pdf


#### Define response format with Controlled Generation
Given the sample marketing campaign brief, we can use Gemini to efficiently extract key details from your past campaign briefs. To take it a step further, we can use [controlled generation](https://cloud.google.com/vertex-ai/generative-ai/docs/multimodal/control-generated-output) which allows you to define a specific schema for the output so that you receive consistently formatted responses. This is particularly useful when you already have an established data schema that you use for other tasks and you'll be able to directly extract data from the model's output without any post-processing.

In the next cell, we define the JSON response schema for our marketing campaign brief.

In [14]:
# JSON response schema for Marketing Campaign Brief


class MarketingCampaignBrief(BaseModel):
    campaign_name: str
    campaign_objectives: list[str]
    target_audience: str
    media_strategy: list[str]
    timeline: str
    target_countries: list[str]
    performance_metrics: list[str]

#### Extract details from sample past campaign brief with Gemini 2.0 Flash

With our response schema all set, we are ready to send our prompt to Gemini 2.0 Flash! As Gemini 2.0 Flash is multimodal, we'll be able to send the PDF document as part of the input for Gemini to process.

When using Vertex AI, you'll be able to pass the file's GCS URL directly to the model instead of having to retrieve and upload it. This makes it convenient for you to build multimodal Gemini-powered apps with seamless integration across Google Cloud. Given Gemini's large context window, GCS provides a scalable and reliable place to store massive datasets, making them readily available for inference.

In the next cell, you'll do the following:
1. Send the prompt together with the sample past campaign brief PDF to Gemini 2.0 Flash
2. Specify that Gemini returns the response in the MarketingCampaignBrief schema you defined previously by including ```response_schema=MarketingCampaignBrief``` in the request

In [15]:
prompt = """
  Extract the details from the sample marketing brief.
"""

marketing_brief_file = types.Part.from_uri(
    file_uri=marketing_brief_file_url, mime_type="application/pdf"
)
contents = [marketing_brief_file, prompt]

response = client.models.generate_content(
    model=MODEL_ID,
    contents=contents,
    config=GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=MarketingCampaignBrief,
    ),
)

sample_marketing_brief = response.text
sample_marketing_brief_json = json.loads(sample_marketing_brief)
print(json.dumps(sample_marketing_brief_json, indent=2))

{
  "campaign_name": "Connect Beyond Limits with Pix Phone 5",
  "campaign_objectives": [
    "Increase awareness of the latest model of the Pix Phone",
    "Generate leads and drive phone sales",
    "Position Pix Phone and the trendy phone to have"
  ],
  "target_audience": "Individuals aged 20-40 in major markets such as US, France, Japan",
  "media_strategy": [
    "Social Media Marketing: Run targeted social media ads on platforms where the target audience is active.",
    "Influencer Marketing: Partner with influencers in the tech industry to promote Pix Phone 5",
    "Paid Advertising: Run targeted display ads on websites and apps frequented by the target audience. Use search engine marketing (SEM) to bid on relevant keywords and appear in search results when potential customers are looking for asset protection insurance."
  ],
  "timeline": "Activity in the 3 major markets in at least the online channels by early Oct 2023. Start from US, followed by France then Japan. The campa

You've successfully extracted the information from the sample past campaign brief PDF document with Gemini 2.0 Flash and Controlled Generation!

### Conduct market research with Google Search as a tool

Next, let's do some market research and find out more about the latest trends in the phone industry so that we can enrich our marketing campaign with the latest information.

But how can I do it if LLMs are frozen in time? Here's where Google Search as a tool can be used, to make Gemini more factual and up-to-date by letting you use Gemini with Google Search in real-time to access and incorporate information from the vast expanse of the public web. You should expect reduced hallucinations and increased accuracy.

**Here are some incredibly useful applications for grounding:**
- Question answering: Get accurate answers to questions that require up-to-date information (e.g., "What's the latest news on...?").
- Content creation: Generate factual and relevant content on various topics.
- Chatbots and conversational AI: Build chatbots that can engage in informed and engaging conversations.

In the next cells, let's use Grounding with Google Search to find out more about the latest trends!

In [16]:
def print_grounding_response(response):
    """Prints Gemini response with grounding citations."""
    grounding_metadata = response.candidates[0].grounding_metadata

    # Citation indices are in byte units
    ENCODING = "utf-8"
    text_bytes = response.text.encode(ENCODING)

    prev_index = 0
    markdown_text = ""

    for grounding_support in grounding_metadata.grounding_supports:
        text_segment = text_bytes[
            prev_index : grounding_support.segment.end_index
        ].decode(ENCODING)

        footnotes_text = ""
        for grounding_chunk_index in grounding_support.grounding_chunk_indices:
            footnotes_text += f"[[{grounding_chunk_index + 1}]]({grounding_metadata.grounding_chunks[grounding_chunk_index].web.uri})\n"

        markdown_text += f"{text_segment} {footnotes_text}\n"
        prev_index = grounding_support.segment.end_index

    if prev_index < len(text_bytes):
        markdown_text += str(text_bytes[prev_index:], encoding=ENCODING)

    markdown_text += "\n----\n## Grounding Sources\n"

    if grounding_metadata.web_search_queries:
        markdown_text += (
            f"\n**Web Search Queries:** {grounding_metadata.web_search_queries}\n"
        )
        if grounding_metadata.search_entry_point:
            markdown_text += f"\n**Search Entry Point:**\n {grounding_metadata.search_entry_point.rendered_content}\n"
    elif grounding_metadata.retrieval_queries:
        markdown_text += (
            f"\n**Retrieval Queries:** {grounding_metadata.retrieval_queries}\n"
        )

    markdown_text += "### Grounding Chunks\n"

    for index, grounding_chunk in enumerate(
        grounding_metadata.grounding_chunks, start=1
    ):
        context = grounding_chunk.web or grounding_chunk.retrieved_context
        if not context:
            print(f"Skipping Grounding Chunk {grounding_chunk}")
            continue

        markdown_text += f"{index}. [{context.title}]({context.uri})\n"

    display(Markdown(markdown_text))

In [17]:
# Use Grounding with Google Search to do market research
market_research_prompt = """
  I am planning to launch a mobile phone campaign and I want to understand the latest trends in the phone industry.
  Please answer the following questions:
  - What are the latest phone models and their selling point from the top 2 phone makers?
  - What is the general public sentiment about mobile phones?
"""

contents = [market_research_prompt]

google_search_tool = Tool(google_search=GoogleSearch())

response = client.models.generate_content(
    model=MODEL_ID,
    contents=contents,
    config=GenerateContentConfig(tools=[google_search_tool]),
)

market_research = response.text
print_grounding_response(response)

Okay, here's an overview of the latest trends in the mobile phone industry for 2026:

**Latest Phone Models and Selling Points (Top Phone Makers)**

It's difficult to definitively name the "top 2" phone makers without concrete market share data for 2026. However, based on 2025 trends, Apple and Samsung are likely contenders [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHI03XWWE3BhlKfoqt73vMjfKys3gAZqIcVmBOn-lHJ4HBW1f-IBuTgvQa6Kv0VZYmRbxnHoSO7Ihtq1sC3CSfGJjc9_IvqxNRK5XYtiSZvwwvX3l8wYGvXYpQOnDGXOhnxjpAlx1PZiQXqp7Dq24RRWPx1U02W7YT13b3QdtYjbVFxhTrvcV1H1ak1PiV8YABbhZgM0jCAjxuWkZgdyQ==)

. Here's what's anticipated from them and other major players:

*   **Apple:**
    *   **iPhone 17 Pro Max:**  Considered the best iPhone, it features the A19 Pro chip and a high-quality camera system [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEQvr4PDExZuNpAhfelTIY7zAqT4dHLIRGEz5GfOPxbvLRaKFcuSrz1AYX_gBdl9St7sqrNlbeH_WMitvOa2q2SBzRhUx9RLPequdfyVTNQ0LiYqqmXLBE7BHT2TZ7WduQagyoz)

. Key selling points are its ease of use and tight integration into the Apple ecosystem [[3]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG1-vi8eOoQAdgBNuvtW_dAnif7CtQXBYIYPEFL_QsZyLLLjPNhN0eJZEE5RqWCE3nMfaG-Xs8FJAClutAy4sGZTuwZS8XgFwit2ImnALEfP2us4I-9ejVTqk-ksErFjTVJs9I2HTQy_EgojTkFOQ9YpD6HzgQrpn0NLL1x3x-yFLTbReHsAr7m70lIzD4zJUF03_sDi4DZbV9LMQ==)

.
    *   **iPhone 18:** Expected in September 2026, it could feature the latest A-series chip, upgraded cameras, and deeper AI integration [[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHmaVkyA3wkk-NNeK3JvvexO5pVw9-ePWsoRq_9SGw_gIrptV6LrR6K9lMbA9V5Ytr7-RaOUmptM_MmdXgtYcjehE-NcRhPmz-igZCF4mDlrdpSP45LXz7OOxo9q8biIOIRDN5CwZTIKvl9n_5gEiokNRL37NdhOmELfqDv)

.
    *   **iPhone 18 Pro and 18 Pro Max**: Expected to keep the dimensions and look of their predecessors, meaning the iPhone 17 Pro and iPhone 17 Pro Max [[3]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG1-vi8eOoQAdgBNuvtW_dAnif7CtQXBYIYPEFL_QsZyLLLjPNhN0eJZEE5RqWCE3nMfaG-Xs8FJAClutAy4sGZTuwZS8XgFwit2ImnALEfP2us4I-9ejVTqk-ksErFjTVJs9I2HTQy_EgojTkFOQ9YpD6HzgQrpn0NLL1x3x-yFLTbReHsAr7m70lIzD4zJUF03_sDi4DZbV9LMQ==)

. The Dynamic Island would see its size reduced thanks to the miniaturization of Face ID components [[3]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG1-vi8eOoQAdgBNuvtW_dAnif7CtQXBYIYPEFL_QsZyLLLjPNhN0eJZEE5RqWCE3nMfaG-Xs8FJAClutAy4sGZTuwZS8XgFwit2ImnALEfP2us4I-9ejVTqk-ksErFjTVJs9I2HTQy_EgojTkFOQ9YpD6HzgQrpn0NLL1x3x-yFLTbReHsAr7m70lIzD4zJUF03_sDi4DZbV9LMQ==)

.

*   **Samsung:**
    *   **Galaxy S25 Ultra:** Praised as a top-tier Android phone with excellent performance, impressive cameras, and innovative AI features. It also includes the S Pen stylus [[5]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGxax50kbLqzOB1_0P4hGQJ7euj3548SiisRNT3hMrcnC0112BryenpbU6i_VUmhU0Oq0LQ5Y0Mpcmo2zhb5cFKVvki-MSRDXhTwSfE_gnshZJdmXkTIl3qmyr4IdxvSkooXiYGLDc=)

.
    *   **Galaxy S26 Series:** Expected to launch in early 2026, potentially including the Galaxy S26, Galaxy S26+, and Galaxy S26 Ultra [[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGfVcWOyenGRKlQ60FiVW7w1kI8vzxtqvNz2dUYv7YQPCE5nhL4mwvr5f2Y3IC5OD9Xo3jr3BXdZEtOxtCCe_eftpcj9PRulxg7kPfFTDVVSnfq8KhFiihOxbMajjod0YgvrS7EKXioe0IZmv2t-c0gn5QmRnsyCtqgYjsr5ck=)

. Rumors suggest slimmer designs, refreshed camera housings, and zoom upgrades [[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHmaVkyA3wkk-NNeK3JvvexO5pVw9-ePWsoRq_9SGw_gIrptV6LrR6K9lMbA9V5Ytr7-RaOUmptM_MmdXgtYcjehE-NcRhPmz-igZCF4mDlrdpSP45LXz7OOxo9q8biIOIRDN5CwZTIKvl9n_5gEiokNRL37NdhOmELfqDv)

.
    *   **Galaxy Z Fold 8 & Z Flip 8:** Samsung is expected to improve its foldable phones in 2026 [[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGfVcWOyenGRKlQ60FiVW7w1kI8vzxtqvNz2dUYv7YQPCE5nhL4mwvr5f2Y3IC5OD9Xo3jr3BXdZEtOxtCCe_eftpcj9PRulxg7kPfFTDVVSnfq8KhFiihOxbMajjod0YgvrS7EKXioe0IZmv2t-c0gn5QmRnsyCtqgYjsr5ck=)

. The Galaxy Z Fold 8 may have a wider outer screen and a thinner hinge and a bigger battery of more than 5,000mAh [[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGfVcWOyenGRKlQ60FiVW7w1kI8vzxtqvNz2dUYv7YQPCE5nhL4mwvr5f2Y3IC5OD9Xo3jr3BXdZEtOxtCCe_eftpcj9PRulxg7kPfFTDVVSnfq8KhFiihOxbMajjod0YgvrS7EKXioe0IZmv2t-c0gn5QmRnsyCtqgYjsr5ck=)

. The Galaxy Z Flip 8 will likely focus on a larger cover screen for better use when the phone is folded [[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGfVcWOyenGRKlQ60FiVW7w1kI8vzxtqvNz2dUYv7YQPCE5nhL4mwvr5f2Y3IC5OD9Xo3jr3BXdZEtOxtCCe_eftpcj9PRulxg7kPfFTDVVSnfq8KhFiihOxbMajjod0YgvrS7EKXioe0IZmv2t-c0gn5QmRnsyCtqgYjsr5ck=)

.

*   **OnePlus:**
    *   **OnePlus 15:** Stands out with exceptional performance and battery life. It is considered the best Android overall [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEQvr4PDExZuNpAhfelTIY7zAqT4dHLIRGEz5GfOPxbvLRaKFcuSrz1AYX_gBdl9St7sqrNlbeH_WMitvOa2q2SBzRhUx9RLPequdfyVTNQ0LiYqqmXLBE7BHT2TZ7WduQagyoz)

.
    *   **OnePlus 16:** There is hope that OnePlus will continue to push things forward with the OnePlus 16 and it could easily be one of the most exciting phones of 2026 [[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGUvF9cu_1UzEeVafG6EtDmPaixb6BLCG9zBvGIG4aIsawAFW3QifSiYbgxBK45EVulErlUiz55chr4xA8t6Aqm-nVFZsnfp9qK9kcNG5oi5jvw9PmFGxn7n-uCj9Nind6HZPfsMsb4TwjU4lbXFUqELAzL-MZXD9cFHIYzbQqQs8i54CNvp5hZK1fr0SfEdgRRbAXmC6SKV1oOIsR7BWDqFn1b-_-ZVXxFS3su)

.
    *   **OnePlus 14:** Expected to launch in January 2026, it's rumored to have a flat 120Hz display, the latest Snapdragon 8 series chip, and super-fast charging [[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHmaVkyA3wkk-NNeK3JvvexO5pVw9-ePWsoRq_9SGw_gIrptV6LrR6K9lMbA9V5Ytr7-RaOUmptM_MmdXgtYcjehE-NcRhPmz-igZCF4mDlrdpSP45LXz7OOxo9q8biIOIRDN5CwZTIKvl9n_5gEiokNRL37NdhOmELfqDv)

.

*   **Google:**
    *   **Pixel 10 Pro XL:** The ultimate camera phone because it's almost impossible to take a bad photo with the Pixel 10 Pro thanks to Google's clever AI smarts and editing [[8]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHadA1wJ6jTwNP2BIegE78212T9JJZFXAoeoh21Qx1iLmvgk_JHdxGKy74DUNUlQwDWQsP1lRCNXua5K-dzk8EYgo-uNAPDhm9buPesIp1l83co4DHpzrztsKll8U1jR5_pRDyT)

.
    *   **Pixel 11 series:** Google is expected to launch the Google Pixel 11 series in October 2026 [[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGfVcWOyenGRKlQ60FiVW7w1kI8vzxtqvNz2dUYv7YQPCE5nhL4mwvr5f2Y3IC5OD9Xo3jr3BXdZEtOxtCCe_eftpcj9PRulxg7kPfFTDVVSnfq8KhFiihOxbMajjod0YgvrS7EKXioe0IZmv2t-c0gn5QmRnsyCtqgYjsr5ck=)

. The Pixel 11 phones are rumoured to use the new Tensor G6 chip which will bring better on-device AI with Gemini 2.0 [[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGfVcWOyenGRKlQ60FiVW7w1kI8vzxtqvNz2dUYv7YQPCE5nhL4mwvr5f2Y3IC5OD9Xo3jr3BXdZEtOxtCCe_eftpcj9PRulxg7kPfFTDVVSnfq8KhFiihOxbMajjod0YgvrS7EKXioe0IZmv2t-c0gn5QmRnsyCtqgYjsr5ck=)

.

**General Public Sentiment About Mobile Phones**

*   **Consumer Sentiment:**
    *   New York’s overall Index of Consumer Sentiment is above the national index [[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFQbHKvPnsO2tyWUuVqlGHrNOwIOjYAkBWlwwWQxHHifPcmDOcEY3GBSV8Xl1rDv1d_kRDjvAXJPBCHcy4Us0FFvBzYC_y05CIJelDbCnlcM6E_sext9Yu6o8CFA4b-gndxSlb7SCjHTOM3TuGFQZMlnvGgDc894n5jANlFlgp9PY9LDZrGHrz_m7gURJLzuB0EmqyorUNm_Q2p2_GNGF10wxR385wI)

.
    *   Consumer sentiment is depressed relative to a year ago, spending intentions are down but remain resilient [[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFQbHKvPnsO2tyWUuVqlGHrNOwIOjYAkBWlwwWQxHHifPcmDOcEY3GBSV8Xl1rDv1d_kRDjvAXJPBCHcy4Us0FFvBzYC_y05CIJelDbCnlcM6E_sext9Yu6o8CFA4b-gndxSlb7SCjHTOM3TuGFQZMlnvGgDc894n5jANlFlgp9PY9LDZrGHrz_m7gURJLzuB0EmqyorUNm_Q2p2_GNGF10wxR385wI)

.
*   **Key Concerns:**
    *   Rising phone prices are a concern [[10]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHQ04bNj14_cCI0vcyYVYTkL9nc6Nezt02gLcB0seetQpI0zH5oO9FixlYqAYzsEZsmCUWRPj01vVMayTmSex9wHXdUzMiICURTNYU8d5iLo4ehK837s6eVpNfTXIWVidBbaxIxMuQ=)

.
    *   Some sources suggest a potential comeback of 4GB RAM in budget phones due to rising prices [[10]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHQ04bNj14_cCI0vcyYVYTkL9nc6Nezt02gLcB0seetQpI0zH5oO9FixlYqAYzsEZsmCUWRPj01vVMayTmSex9wHXdUzMiICURTNYU8d5iLo4ehK837s6eVpNfTXIWVidBbaxIxMuQ=)

.
    *   Consumers are pessimistic about the broader economy, yet they continue to spend [[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFQbHKvPnsO2tyWUuVqlGHrNOwIOjYAkBWlwwWQxHHifPcmDOcEY3GBSV8Xl1rDv1d_kRDjvAXJPBCHcy4Us0FFvBzYC_y05CIJelDbCnlcM6E_sext9Yu6o8CFA4b-gndxSlb7SCjHTOM3TuGFQZMlnvGgDc894n5jANlFlgp9PY9LDZrGHrz_m7gURJLzuB0EmqyorUNm_Q2p2_GNGF10wxR385wI)

.
*   **Positive Trends:**
    *   Increasing premium demand and the adoption of 5G are driving market growth [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHI03XWWE3BhlKfoqt73vMjfKys3gAZqIcVmBOn-lHJ4HBW1f-IBuTgvQa6Kv0VZYmRbxnHoSO7Ihtq1sC3CSfGJjc9_IvqxNRK5XYtiSZvwwvX3l8wYGvXYpQOnDGXOhnxjpAlx1PZiQXqp7Dq24RRWPx1U02W7YT13b3QdtYjbVFxhTrvcV1H1ak1PiV8YABbhZgM0jCAjxuWkZgdyQ==)

.
    *   Software experiences, including AI, are becoming a key differentiator [[10]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHQ04bNj14_cCI0vcyYVYTkL9nc6Nezt02gLcB0seetQpI0zH5oO9FixlYqAYzsEZsmCUWRPj01vVMayTmSex9wHXdUzMiICURTNYU8d5iLo4ehK837s6eVpNfTXIWVidBbaxIxMuQ=)

.
*   **Potential Issues:**
    *   Gen Z might be reluctant to engage in face-to-face conversations, opting for texting [[11]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGG6AVEjMyK9fAlc3gTvrWbhgpThxdcv_vYNm9Ibk5nQIpDNlYfiA6t6MGi6vVi0n8rhoFVvNzxas4wk-PA5zPtFrSkY7VE1K6zMekV_VNZZdn6ENlkkamCpumStnkZPmEUyz0W8xLztftOnLONmhmSVZAuC6ZlPHHnhgpkKZs4SXpXFRio9q3DekkMuayCuGPCVFZt)

.
    *   The pandemic may have reduced opportunities for young people to practice socializing [[11]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGG6AVEjMyK9fAlc3gTvrWbhgpThxdcv_vYNm9Ibk5nQIpDNlYfiA6t6MGi6vVi0n8rhoFVvNzxas4wk-PA5zPtFrSkY7VE1K6zMekV_VNZZdn6ENlkkamCpumStnkZPmEUyz0W8xLztftOnLONmhmSVZAuC6ZlPHHnhgpkKZs4SXpXFRio9q3DekkMuayCuGPCVFZt)

.

----
## Grounding Sources

**Web Search Queries:** ['latest mobile phone models 2026', 'top 2 phone makers 2026', 'public sentiment mobile phones 2026', 'most popular phones 2026']

**Search Entry Point:**
 <style>
.container {
  align-items: center;
  border-radius: 8px;
  display: flex;
  font-family: Google Sans, Roboto, sans-serif;
  font-size: 14px;
  line-height: 20px;
  padding: 8px 12px;
}
.chip {
  display: inline-block;
  border: solid 1px;
  border-radius: 16px;
  min-width: 14px;
  padding: 5px 16px;
  text-align: center;
  user-select: none;
  margin: 0 8px;
  -webkit-tap-highlight-color: transparent;
}
.carousel {
  overflow: auto;
  scrollbar-width: none;
  white-space: nowrap;
  margin-right: -12px;
}
.headline {
  display: flex;
  margin-right: 4px;
}
.gradient-container {
  position: relative;
}
.gradient {
  position: absolute;
  transform: translate(3px, -9px);
  height: 36px;
  width: 9px;
}
@media (prefers-color-scheme: light) {
  .container {
    background-color: #fafafa;
    box-shadow: 0 0 0 1px #0000000f;
  }
  .headline-label {
    color: #1f1f1f;
  }
  .chip {
    background-color: #ffffff;
    border-color: #d2d2d2;
    color: #5e5e5e;
    text-decoration: none;
  }
  .chip:hover {
    background-color: #f2f2f2;
  }
  .chip:focus {
    background-color: #f2f2f2;
  }
  .chip:active {
    background-color: #d8d8d8;
    border-color: #b6b6b6;
  }
  .logo-dark {
    display: none;
  }
  .gradient {
    background: linear-gradient(90deg, #fafafa 15%, #fafafa00 100%);
  }
}
@media (prefers-color-scheme: dark) {
  .container {
    background-color: #1f1f1f;
    box-shadow: 0 0 0 1px #ffffff26;
  }
  .headline-label {
    color: #fff;
  }
  .chip {
    background-color: #2c2c2c;
    border-color: #3c4043;
    color: #fff;
    text-decoration: none;
  }
  .chip:hover {
    background-color: #353536;
  }
  .chip:focus {
    background-color: #353536;
  }
  .chip:active {
    background-color: #464849;
    border-color: #53575b;
  }
  .logo-light {
    display: none;
  }
  .gradient {
    background: linear-gradient(90deg, #1f1f1f 15%, #1f1f1f00 100%);
  }
}
</style>
<div class="container">
  <div class="headline">
    <svg class="logo-light" width="18" height="18" viewBox="9 9 35 35" fill="none" xmlns="http://www.w3.org/2000/svg">
      <path fill-rule="evenodd" clip-rule="evenodd" d="M42.8622 27.0064C42.8622 25.7839 42.7525 24.6084 42.5487 23.4799H26.3109V30.1568H35.5897C35.1821 32.3041 33.9596 34.1222 32.1258 35.3448V39.6864H37.7213C40.9814 36.677 42.8622 32.2571 42.8622 27.0064V27.0064Z" fill="#4285F4"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M26.3109 43.8555C30.9659 43.8555 34.8687 42.3195 37.7213 39.6863L32.1258 35.3447C30.5898 36.3792 28.6306 37.0061 26.3109 37.0061C21.8282 37.0061 18.0195 33.9811 16.6559 29.906H10.9194V34.3573C13.7563 39.9841 19.5712 43.8555 26.3109 43.8555V43.8555Z" fill="#34A853"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M16.6559 29.8904C16.3111 28.8559 16.1074 27.7588 16.1074 26.6146C16.1074 25.4704 16.3111 24.3733 16.6559 23.3388V18.8875H10.9194C9.74388 21.2072 9.06992 23.8247 9.06992 26.6146C9.06992 29.4045 9.74388 32.022 10.9194 34.3417L15.3864 30.8621L16.6559 29.8904V29.8904Z" fill="#FBBC05"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M26.3109 16.2386C28.85 16.2386 31.107 17.1164 32.9095 18.8091L37.8466 13.8719C34.853 11.082 30.9659 9.3736 26.3109 9.3736C19.5712 9.3736 13.7563 13.245 10.9194 18.8875L16.6559 23.3388C18.0195 19.2636 21.8282 16.2386 26.3109 16.2386V16.2386Z" fill="#EA4335"/>
    </svg>
    <svg class="logo-dark" width="18" height="18" viewBox="0 0 48 48" xmlns="http://www.w3.org/2000/svg">
      <circle cx="24" cy="23" fill="#FFF" r="22"/>
      <path d="M33.76 34.26c2.75-2.56 4.49-6.37 4.49-11.26 0-.89-.08-1.84-.29-3H24.01v5.99h8.03c-.4 2.02-1.5 3.56-3.07 4.56v.75l3.91 2.97h.88z" fill="#4285F4"/>
      <path d="M15.58 25.77A8.845 8.845 0 0 0 24 31.86c1.92 0 3.62-.46 4.97-1.31l4.79 3.71C31.14 36.7 27.65 38 24 38c-5.93 0-11.01-3.4-13.45-8.36l.17-1.01 4.06-2.85h.8z" fill="#34A853"/>
      <path d="M15.59 20.21a8.864 8.864 0 0 0 0 5.58l-5.03 3.86c-.98-2-1.53-4.25-1.53-6.64 0-2.39.55-4.64 1.53-6.64l1-.22 3.81 2.98.22 1.08z" fill="#FBBC05"/>
      <path d="M24 14.14c2.11 0 4.02.75 5.52 1.98l4.36-4.36C31.22 9.43 27.81 8 24 8c-5.93 0-11.01 3.4-13.45 8.36l5.03 3.85A8.86 8.86 0 0 1 24 14.14z" fill="#EA4335"/>
    </svg>
    <div class="gradient-container"><div class="gradient"></div></div>
  </div>
  <div class="carousel">
    <a class="chip" href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGZfjemi4CVQ4Akn37Z8TBHIzqyRCFDQ_kIWWzqjR4pxgLrM6xJBuodTLXwdKPC3mmmsrOE334ztIBu6bFOIYT0Wnn8gvrwclaZq8qwNERP7CbmRUxBlR53X0iRP7h-iRaJSogRbsTJu1xHS1xc4Drvq7c22KXfxeapQ8XnyPomcQ-bw063kXdgzaLmdEDrttJ87IU_XgqVvzkqnw==">most popular phones 2026</a>
    <a class="chip" href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEmrPRz1nA2p-TlQfymbFKgVEy5potY5u55hvHZG-wv2YwPqRFJ_x-41KrZLMEOfBpvOdlH_kNNxG3hiMf786OwEGuJj_hJ_YXKh4bBm-v4PQmxFPDkW-164oH1Zcj9b3vOzJ07u5fWIT1E57spMA2TI-_gRbq4yIDiFWncmarVUbbPMBLL0cD48493Rsk1LlyiJME3BG89lepy-AaAa8QE37uKgiZ3">public sentiment mobile phones 2026</a>
    <a class="chip" href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGZOA9LU1BILAMCVkW-U0nAhaIv3stPHH_3-ekMY1kWmZ9sAg7WNI5_RDFOsl77LDBScp3yaEv5q5haRlEQZw8G_F8BIizxyIdmd4JGdZ3THBT1k8vEeOyNv5Lo4xdndtELSESx3DBixjmci6d_ciou1mVtfAsECX_ZVoZOpY2_PzVxQKxcF0_aZyO2TuJtL7Bi7XtGbU3VxdjeoOmi86QAX0M=">latest mobile phone models 2026</a>
    <a class="chip" href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEql3XlIvEVnI12UwEhJoWaL8lDOVRuLBlpuzC5fp2NNZkqAqrwjBt4kYdfixE50AMMR85cJHv1FvvNlSg2OCMWezHmz9NI9k88q3kT4uHusOLuJdvR3SCcKqZBGbjW3s18NVNT1EF04Mpzdl2v-yYCOg-BIYhZmkAG3lEjfj-opSAEXC0cCRfc5iaYySWWtYaUExfC_GV_9_pG">top 2 phone makers 2026</a>
  </div>
</div>

### Grounding Chunks
1. [counterpointresearch.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHI03XWWE3BhlKfoqt73vMjfKys3gAZqIcVmBOn-lHJ4HBW1f-IBuTgvQa6Kv0VZYmRbxnHoSO7Ihtq1sC3CSfGJjc9_IvqxNRK5XYtiSZvwwvX3l8wYGvXYpQOnDGXOhnxjpAlx1PZiQXqp7Dq24RRWPx1U02W7YT13b3QdtYjbVFxhTrvcV1H1ak1PiV8YABbhZgM0jCAjxuWkZgdyQ==)
2. [techradar.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEQvr4PDExZuNpAhfelTIY7zAqT4dHLIRGEz5GfOPxbvLRaKFcuSrz1AYX_gBdl9St7sqrNlbeH_WMitvOa2q2SBzRhUx9RLPequdfyVTNQ0LiYqqmXLBE7BHT2TZ7WduQagyoz)
3. [medium.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG1-vi8eOoQAdgBNuvtW_dAnif7CtQXBYIYPEFL_QsZyLLLjPNhN0eJZEE5RqWCE3nMfaG-Xs8FJAClutAy4sGZTuwZS8XgFwit2ImnALEfP2us4I-9ejVTqk-ksErFjTVJs9I2HTQy_EgojTkFOQ9YpD6HzgQrpn0NLL1x3x-yFLTbReHsAr7m70lIzD4zJUF03_sDi4DZbV9LMQ==)
4. [mozillion.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHmaVkyA3wkk-NNeK3JvvexO5pVw9-ePWsoRq_9SGw_gIrptV6LrR6K9lMbA9V5Ytr7-RaOUmptM_MmdXgtYcjehE-NcRhPmz-igZCF4mDlrdpSP45LXz7OOxo9q8biIOIRDN5CwZTIKvl9n_5gEiokNRL37NdhOmELfqDv)
5. [pcmag.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGxax50kbLqzOB1_0P4hGQJ7euj3548SiisRNT3hMrcnC0112BryenpbU6i_VUmhU0Oq0LQ5Y0Mpcmo2zhb5cFKVvki-MSRDXhTwSfE_gnshZJdmXkTIl3qmyr4IdxvSkooXiYGLDc=)
6. [cashify.in](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGfVcWOyenGRKlQ60FiVW7w1kI8vzxtqvNz2dUYv7YQPCE5nhL4mwvr5f2Y3IC5OD9Xo3jr3BXdZEtOxtCCe_eftpcj9PRulxg7kPfFTDVVSnfq8KhFiihOxbMajjod0YgvrS7EKXioe0IZmv2t-c0gn5QmRnsyCtqgYjsr5ck=)
7. [techradar.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGUvF9cu_1UzEeVafG6EtDmPaixb6BLCG9zBvGIG4aIsawAFW3QifSiYbgxBK45EVulErlUiz55chr4xA8t6Aqm-nVFZsnfp9qK9kcNG5oi5jvw9PmFGxn7n-uCj9Nind6HZPfsMsb4TwjU4lbXFUqELAzL-MZXD9cFHIYzbQqQs8i54CNvp5hZK1fr0SfEdgRRbAXmC6SKV1oOIsR7BWDqFn1b-_-ZVXxFS3su)
8. [stuff.tv](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHadA1wJ6jTwNP2BIegE78212T9JJZFXAoeoh21Qx1iLmvgk_JHdxGKy74DUNUlQwDWQsP1lRCNXua5K-dzk8EYgo-uNAPDhm9buPesIp1l83co4DHpzrztsKll8U1jR5_pRDyT)
9. [siena.edu](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFQbHKvPnsO2tyWUuVqlGHrNOwIOjYAkBWlwwWQxHHifPcmDOcEY3GBSV8Xl1rDv1d_kRDjvAXJPBCHcy4Us0FFvBzYC_y05CIJelDbCnlcM6E_sext9Yu6o8CFA4b-gndxSlb7SCjHTOM3TuGFQZMlnvGgDc894n5jANlFlgp9PY9LDZrGHrz_m7gURJLzuB0EmqyorUNm_Q2p2_GNGF10wxR385wI)
10. [youtube.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHQ04bNj14_cCI0vcyYVYTkL9nc6Nezt02gLcB0seetQpI0zH5oO9FixlYqAYzsEZsmCUWRPj01vVMayTmSex9wHXdUzMiICURTNYU8d5iLo4ehK837s6eVpNfTXIWVidBbaxIxMuQ=)
11. [washingtonpost.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGG6AVEjMyK9fAlc3gTvrWbhgpThxdcv_vYNm9Ibk5nQIpDNlYfiA6t6MGi6vVi0n8rhoFVvNzxas4wk-PA5zPtFrSkY7VE1K6zMekV_VNZZdn6ENlkkamCpumStnkZPmEUyz0W8xLztftOnLONmhmSVZAuC6ZlPHHnhgpkKZs4SXpXFRio9q3DekkMuayCuGPCVFZt)


Congratulations on using Grounding with Google Search to get the latest information around the phone industry in your market research!

### Putting it together

Now that we have a template and some market research done, let's try to create a marketing brief for our new phone launch with Gemini 2.0 Flash!

In the next cell, you'll pass the following information to Gemini 2.0 Flash:
1. Information about the phone that you're launching
2. Prompt to instruct Gemini to create a marketing campaign brief
3. Extracted information from the sample past campaign brief
4. Market research that was done with Grounding with Google Search
5. MarketingCampaignBrief schema that was defined previously


In [18]:
new_phone_details = """
  Phone Name: Pix Phone 10
  Short description: Pix Phone 10 is the flagship phone with a focus on AI-powered features and a completely redesigned form factor.
  Tech Specs:
    - Camera: 50MP main sensor with 48MP ultrawide lens with autofocus for macro shots
    - Performance: P5 processor for fast performance and AI capabilities
    - Battery: 4700mAh battery for all-day usage
  Key Highlights:
    - Powerful camera system
    - Redesigned software user experience to introduce more fun
    - Compact form factor
  Launch timeline: Jan 2025
  Target countries: US, France and Japan
"""

create_brief_prompt = f"""
Given the following details, create a marketing campaign brief for the new phone launch:

Sample campaign brief:
{sample_marketing_brief}

Market research:
{market_research}

New phone details:
{new_phone_details}
"""

contents = [create_brief_prompt]

response = client.models.generate_content(
    model=MODEL_ID,
    contents=contents,
    config=GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=MarketingCampaignBrief,
    ),
)

creative_brief = response.text
creative_brief_json = json.loads(creative_brief)
print(json.dumps(creative_brief_json, indent=2))

{
  "campaign_name": "Experience the Future with Pix Phone 10",
  "campaign_objectives": [
    "Generate excitement and awareness for the launch of Pix Phone 10",
    "Drive pre-orders and initial sales of Pix Phone 10",
    "Establish Pix Phone 10 as the leading AI-powered phone with a fun user experience"
  ],
  "target_audience": "Tech-savvy individuals aged 22-35 in US, France, and Japan who value innovation, camera quality, and user experience.",
  "media_strategy": [
    "Social Media Marketing: Launch engaging campaigns on platforms like Instagram, TikTok, and YouTube, showcasing the phone's camera capabilities and AI features.",
    "Influencer Marketing: Partner with tech reviewers and lifestyle influencers to create authentic content highlighting the phone's unique features.",
    "Online Advertising: Utilize targeted display ads and search engine marketing to reach potential customers actively searching for new phones.",
    "Public Relations: Secure reviews and coverage fro

You've successfully created your marketing campaign brief for your upcoming phone launch!

### Creating Assets for the Marketing Campaign
Now that we have our marketing campaign brief for the upcoming phone launch, we can now use it as information and context to generate some marketing assets.

Gemini supports a variety of languages ([complete list](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/models#languages-gemini)) so it is perfect for us as we will need to generate assets in the local language of our target markets: US, France and Japan.

In the following sections, we will be looking at creating:
- Social Media Ad Copy
- Storyboarding for short-form videos

#### Creating Social Media Ad Copy
Similarly, we will be defining a response schema for ad copy and passing it it to Gemini 2.0 Flash in the request.

In the next few cells, we will do the following:
1. Define the JSON response schema for our ad copy
2. Send the prompt and response schema to Gemini 2.0 Flash

In [19]:
# JSON response schema for an ad copy


class AdCopy(BaseModel):
    ad_copy_options: list[str]
    localization_notes: list[str]
    visual_description: list[str]

In [20]:
ad_copy_prompt = f"""
  Given the marketing campaign brief, create an Instagram ad-copy for each target market: {creative_brief_json["target_countries"]}
  Please localize the ad-copy and the visuals to the target markets for better relevancy to the target audience.
  Marketing Campaign Brief:
  {creative_brief}
"""

contents = [ad_copy_prompt]

response = client.models.generate_content(
    model=MODEL_ID,
    contents=contents,
    config=GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=AdCopy,
    ),
)

ad_copy = response.text
ad_copy_json = json.loads(ad_copy)
print(json.dumps(ad_copy_json, indent=2, ensure_ascii=False))

{
  "ad_copy_options": [
    "🇺🇸 US: Experience the future in your hands with the all-new Pix Phone 10! 🚀 Capture stunning photos with our revolutionary AI camera and enjoy a seamless, intuitive user experience. Pre-order now and be among the first to own the AI-powered Pix Phone 10! #PixPhone10 #AIcamera #Innovation #Tech",
    "🇫🇷 France: Découvrez le futur entre vos mains avec le tout nouveau Pix Phone 10 ! 🚀 Capturez des photos exceptionnelles avec notre appareil photo IA révolutionnaire et profitez d'une expérience utilisateur fluide et intuitive. Précommandez maintenant et soyez parmi les premiers à posséder le Pix Phone 10 doté de l'IA ! #PixPhone10 #IAcamera #Innovation #Tech",
    "🇯🇵 Japan: 全く新しいPix Phone 10で、未来を手にしましょう！🚀 革新的なAIカメラで素晴らしい写真を撮影し、シームレスで直感的なユーザーエクスペリエンスをお楽しみください。 今すぐ予約注文して、AI搭載のPix Phone 10をいち早く手に入れましょう！ #PixPhone10 #AIカメラ #イノベーション #テック"
  ],
  "localization_notes": [
    "US: Ad copy focuses on excitement and ownership. Hashtags are in English.",
    "France: Ad

You've successfully created localized ad copies for each of the target markets in its respective local language!

#### Creating storyboard for short-form videos
Lastly, let's get Gemini 2.0 Flash to help us brainstorm a storyboard for a short-form video to accompany the phone launch campaign!


In [21]:
short_video_prompt = f"""
  Given the marketing campaign brief, create a storyboard for a YouTube Shorts video for target markets: {creative_brief_json["target_countries"]}.
  Please localize the content to the target markets for better relevancy to the target audience.
  Marketing Campaign Brief:
  {creative_brief}

"""

contents = [short_video_prompt]

response = client.models.generate_content(model=MODEL_ID, contents=contents)

short_video_response = response.text
display(Markdown(short_video_response))

Okay, here's a storyboard outline for a YouTube Shorts video for the "Experience the Future with Pix Phone 10" campaign, localized for the US, France, and Japan. Each country will have its own slightly different version, keeping the core message consistent but adapting the visuals and music.

**Core Concept:**  The video showcases the seamless AI-powered user experience of the Pix Phone 10 in everyday scenarios, highlighting its speed, camera quality, and ease of use. The style will be fast-paced, visually appealing, and fun.

**General Structure:**

*   **Intro (1-2 seconds):** Eye-catching visual hook related to the key feature being showcased.
*   **Problem (1-2 seconds):** Briefly demonstrate a common frustration the Pix Phone 10 solves.
*   **Solution (3-4 seconds):** Showcase the Pix Phone 10 solving the problem with its AI capabilities and easy UX.
*   **Benefit (2-3 seconds):** Highlight the positive outcome and joy of using the phone.
*   **Call to Action (1-2 seconds):**  Encourage viewers to learn more and pre-order.

**Visual Style:**

*   Clean, modern aesthetic.
*   Bright, vibrant colors (but adjusted based on the target market - US might be bolder, France more chic, Japan cuter).
*   Emphasis on close-up shots of the phone's screen and camera.
*   Use of text overlays to highlight key features and benefits.
*   Smooth transitions and engaging visual effects.

**Music:**

*   Upbeat, trendy track that reflects the youthfulness of the target audience.  Again, localized to the specific music trends of each country.

**Sound Design:**

*   Clean and crisp sound effects that complement the visuals and enhance the user experience.
*   Use of voiceovers where appropriate, delivered by relatable and engaging voices.

**Storyboard Outline (Template - Country-Specific Details Follow):**

| **Scene #** | **Visual Description**                                                                                                           | **Audio Description**                                                                                                                                                                                                                              | **Text Overlay**                                   | **Duration** |
| :---------- | :---------------------------------------------------------------------------------------------------------------------------- | :-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | :------------------------------------------------- | :----------- |
| 1           | **Close-up of someone's hand fumbling with a cluttered phone screen.**                                                      | **Frustrated sigh or groan.** Sound of notification overload.                                                                                                                                                                                |  "Tired of Phone Chaos?"                           | 1 sec        |
| 2           | **Transition to a hand smoothly navigating the Pix Phone 10's clean, intuitive interface. Focus on AI-powered organization.** | **Upbeat, futuristic sound effect. Gentle chime indicating AI assistance.**                                                                                                                                                                   | "AI to the Rescue!"/ "Effortless Organization" | 2 secs        |
| 3           | **Showcase the phone's camera:  A quick montage of vibrant photos and videos taken with the Pix Phone 10 in diverse settings.** | **Camera shutter sounds, upbeat music continues, maybe a subtle "wow" sound effect.**                                                                                                                                                           | "Pro-Level Camera"                                 | 2 secs        |
| 4           | **A person effortlessly using the phone's AI assistant to quickly translate a sign/menu in a foreign language.**                 | **Sound of seamless translation happening in real-time.  Voiceover (optional): "Translate Anything, Anywhere."**                                                                                                                          | "Live Translation"                                  | 2 secs        |
| 5           | **Smiling person enjoying a perfectly edited photo/video thanks to the phone's AI editing tools.**                               | **Happy, satisfied sigh.  Music swells slightly.**                                                                                                                                                                                          | "Perfectly Edited, Instantly"                       | 2 secs        |
| 6           | **Screen showing Pix Phone 10 website. Call to action to pre-order.**                                                      | **Upbeat music fades slightly. Voiceover (Optional): "Experience the Future. Pre-order your Pix Phone 10 today!"** or  Just music with sound effect chimes                                                                                   | "Pre-order NOW!"   Website URL                     | 2 secs        |

---

**Country-Specific Localizations:**

**1. US Version:**

*   **Scene 1:**  Show someone overwhelmed with apps and notifications while waiting in line at a coffee shop.
*   **Scene 2:** Focus on the AI's ability to filter out spam calls and organize emails. Use American slang like "No more spam headaches!"
*   **Scene 3:** Show diverse scenes - a bustling city street, a scenic hiking trail, a backyard BBQ.  Music should be current US Top 40 or a popular indie/pop track.
*   **Scene 4:**  Translation used to order food from a food truck with diverse cuisine.
*   **Scene 5:**  Person posting the edited photo to Instagram and getting lots of likes.
*   **Text Overlays:** Use American English slang and colloquialisms. Focus on convenience and speed.
*   **Call to Action:** "Get Yours First! Pre-Order Now!"

**2. France Version:**

*   **Scene 1:** Show someone struggling to manage social media, emails, and calendar notifications on a stylish Parisian cafe.
*   **Scene 2:** Highlight the AI's ability to schedule meetings and manage a busy social life.  Use sophisticated French phrases.
*   **Scene 3:** Show scenes of Parisian landmarks, art galleries, and fashionable cafes.  Music should be French indie-pop or electronic music.
*   **Scene 4:** Translation used to read a menu at a chic restaurant or understand a local artisan's description of their craft.
*   **Scene 5:** Person sharing the perfectly edited photo on a social media platform, with a focus on aesthetic and elegance.
*   **Text Overlays:** Use elegant French phrasing. Emphasize style and sophistication.
*   **Call to Action:** "Réservez le vôtre ! Précommandez maintenant !" (Reserve yours! Pre-order now!)

**3. Japan Version:**

*   **Scene 1:** Show someone struggling to keep up with a flood of LINE messages and app notifications on a crowded train.
*   **Scene 2:** Focus on the AI's ability to prioritize important messages and filter out irrelevant information. Use polite Japanese language.
*   **Scene 3:** Show scenes of vibrant Tokyo streets, cherry blossom viewing, and delicious Japanese food. Music should be J-Pop or anime-inspired electronic music.
*   **Scene 4:** Translation used to understand instructions for using a vending machine or navigating a complex train station.
*   **Scene 5:** Person sharing the perfectly edited photo on a popular Japanese social media platform, with a focus on cuteness and perfection.
*   **Text Overlays:** Use clear and concise Japanese text with appropriate honorifics.  Emphasize efficiency and ease of use.  Consider using kawaii (cute) fonts and icons.
*   **Call to Action:** "今すぐ予約！プリオーダーはこちら！" (Reserve now! Pre-order here!)

**Important Considerations for Localization:**

*   **Casting:**  Use actors who are representative of the target audience in each country.
*   **Language:**  Ensure accurate and natural-sounding translations.  Consider using native speakers for voiceovers.
*   **Cultural Nuances:**  Be aware of cultural sensitivities and avoid any potentially offensive imagery or messaging.
*   **Platforms:** While this is for YouTube shorts, understand that cross-promotion across other platforms such as TikTok and Instagram will require understanding the nuances of those respective channels in each region to promote the content.
*   **Trends:** Be mindful of current social media trends and incorporate them into the video where appropriate.

This storyboard provides a solid foundation for creating a highly engaging and effective YouTube Shorts campaign for the Pix Phone 10. Remember to test and iterate based on performance data to maximize the campaign's impact. Good luck!


There you go! You've successfully created storyboards for each of the target markets - localized to the local context!

# Conclusion
In this tutorial, you've learned:
- How to use the unified Google Gen AI SDK
- Extract information from PDF documents
- Use Controlled Generation to ensure consistent output in specified schema
- Utilize Grounding with Google Search to access latest information
- Create a marketing campaign and assets with Gemini